# Tables IV and XII — zero-shot coverage

## Standalone reproduction

This notebook is fully standalone. It does **not** depend on another TGCM notebook, the TGCM model code, checkpoints, or outputs from another experiment.

### Environment

- Python 3.12
- pandas 2.2
- JupyterLab / IPython
- CPU only; CUDA is not required.

For this notebook alone, `pip install pandas==2.2.*` is sufficient.

### Required data

- `kill_chain_mapping.json` — disclosed technique-to-kill-chain mapping.

The mapping is downloaded automatically from a pinned TGCM_Website commit. No checkpoint, CAPture raw CSV, GPU, API key, or proprietary service is required.

### Output

The notebook expands the mapping into the detailed technique/phase table (Table XII) and derives grouped coverage counts (Table IV).

In [ ]:
from pathlib import Path
import json
import urllib.request
import pandas as pd

SUPPORT_REV = "00f786078cf79f01fe2398cc57b7f8e0c45c70fc"
MAPPING_URL = (
    "https://raw.githubusercontent.com/Irish-kw/TGCM_Website/"
    + SUPPORT_REV
    + "/reproduction/paper_metadata/kill_chain_mapping.json"
)
MAPPING_FILE = Path.cwd() / "kill_chain_mapping.json"

if not MAPPING_FILE.is_file():
    print("Downloading disclosed mapping...")
    urllib.request.urlretrieve(MAPPING_URL, MAPPING_FILE)

mapping_payload = json.loads(MAPPING_FILE.read_text(encoding="utf-8"))
print(f"Loaded mapping for {len(mapping_payload)} datasets from {MAPPING_FILE}")

In [ ]:
detail_rows = []
for dataset, phases in mapping_payload.items():
    for phase, techniques in phases.items():
        for technique in techniques:
            detail_rows.append({
                "dataset": dataset,
                "kill_chain_phase": phase,
                "technique": technique,
            })

mapping = pd.DataFrame(detail_rows)
phase_order = list(next(iter(mapping_payload.values())).keys())
coverage = (
    mapping.groupby(["dataset", "kill_chain_phase"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=list(mapping_payload), columns=phase_order, fill_value=0)
    .reset_index()
)
coverage

In [ ]:
mapping